# Kaggle Playground Series S6E9: Predicting Electric Vehicle Purchases

This notebook provides the exploratory data analysis, feature engineering, and model training pipeline using XGBoost with 5-fold Stratified K-Fold validation.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load datasets
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

In [ ]:
# Preprocessing & Feature Engineering
y = (train['Will_Buy_EV'] == 'Yes').astype(int)
train_id = train['id']
test_id = test['id']

train = train.drop(['Will_Buy_EV', 'id'], axis=1)
test = test.drop(['id'], axis=1)

def preprocess(df):
    df_mod = df.copy()
    subsidy_num = df_mod['Subsidy_Available'].map({'No': 0, 'Yes': 1})
    df_mod['Subsidy_x_Income'] = subsidy_num * df_mod['Annual_Income_USD']
    cat_cols = df_mod.select_dtypes(include=['object']).columns.tolist()
    for c in cat_cols:
        df_mod[c] = df_mod[c].astype('category')
    return df_mod

X = preprocess(train)
X_test = preprocess(test)

In [ ]:
# Model Training & 5-Fold Stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y))
best_iters = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    model = XGBClassifier(
        random_state=42, enable_categorical=True, tree_method='hist', n_jobs=-1,
        max_depth=5, learning_rate=0.10, reg_lambda=10, min_child_weight=1, n_estimators=1500,
        early_stopping_rounds=50
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    best_iter = model.best_iteration
    best_iters.append(best_iter)
    preds = model.predict_proba(X_va, iteration_range=(0, best_iter + 1))[:, 1]
    oof_preds[val_idx] = preds
    print(f'Fold {fold+1} | AUC: {roc_auc_score(y_va, preds):.6f}')

print(f'Overall OOF AUC: {roc_auc_score(y, oof_preds):.6f}')